In [ ]:
# CROSSSILO_ALIGN — CELL 1: setup, alignment harness, exact-reproduction check, smoke test. GPU T4, ~8-10 min.
#   New notebook for WP3 (domain alignment). Reads the existing cross-silo data and harness READ-ONLY;
#   writes only to results/align/ (code copy, check runs). Prints relative paths only.
#   Methods in the harness (all cross-silo FedAvg on the same 20 clients, seeds and splits as the cross-silo grid):
#     fedavg      : plain FedAvg, no penalty (used for the exact-reproduction check)
#     coral       : CE + lam * CORAL distance between the client's batch covariance of the 128-unit hidden layer
#                   and the OTHER silo's covariance (shared as an aggregate statistic each round; no raw flows leave a silo)
#     mmd         : CE + lam * MMD^2 with a Gaussian kernel, approximated by 512 random Fourier features; the other
#                   silo's mean feature embedding is shared each round (aggregate statistic)
#     bn / fedbn  : FA-MLP with batch normalization; BN shared by all clients (bn) or kept silo-local (fedbn)
import os, glob, json, time, sys, importlib, gc
import numpy as np, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; AL = f'{DRIVE_PATH}/results/align'
for sub in ('code', 'check', 'smoke'): os.makedirs(f'{AL}/{sub}', exist_ok=True)
LR = json.load(open(f'{XS}/lr_chosen.json'))
src = open(f'{V2D}/fl_core.py').read(); assert src.count('N_CLASSES = 8') == 1
open('/content/fl_xp.py', 'w').write(src.replace('N_CLASSES = 8', 'N_CLASSES = 5'))

ALIGN_CORE = r'''
"""align_core.py - cross-silo FedAvg with optional domain alignment (CORAL, RFF-MMD) or batch normalization (shared / silo-local).
Same clients, initial weights, local optimizer (SGD momentum 0.9 reset per client), clipping and evaluation as the cross-silo grid."""
import time, gc, numpy as np, tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import f1_score, confusion_matrix
NC = 5; LAB = np.arange(NC)
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=LAB))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=LAB)]

def build_bn_model(n_feat):
    inp = layers.Input(shape=(n_feat,), name='x')
    a = layers.Softmax(name='feat_attention')(layers.Dense(n_feat, name='feat_attn_score')(inp))
    x = layers.Lambda(lambda t: t[0] * t[1] * n_feat, name='attn_reweight')([inp, a])
    h = layers.Dense(256)(x); h = layers.BatchNormalization()(h); h = layers.Activation('relu')(h); h = layers.Dropout(0.2)(h)
    h = layers.Dense(128)(h); h = layers.BatchNormalization()(h); h = layers.Activation('relu', name='h128')(h)
    return Model(inp, layers.Dense(NC, activation='softmax')(h))

def hidden_tensor(model, bn):
    if bn: return model.get_layer('h128').output
    d128 = [l for l in model.layers if isinstance(l, layers.Dense) and l.units == 128]; assert len(d128) == 1
    return d128[0].output

def run_align(DATA, fl_xp, setting, method, seed, lr, lam=0.0, bn=False, bn_local=False, rounds=100, batch=256,
              momentum=0.9, clip_norm=1.0, eval_every=10, alpha=0.5, n_rff=512, log=None):
    doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
    assert method in ('fedavg', 'coral', 'mmd') and (method == 'fedavg' or setting == 'xsilo')
    assert not bn_local or (bn and setting == 'xsilo')
    X = np.concatenate([DATA[d]['X_train'] for d in doms]); y = np.concatenate([DATA[d]['y_train'] for d in doms])
    parts, silo, off = [], [], 0
    for i, d in enumerate(doms):                                   # identical clients to the cross-silo grid
        ps = fl_xp.partition(DATA[d]['y_train'], 'dirichlet', 10, seed, alpha=alpha)
        parts += [p + off for p in ps]; silo += [i] * len(ps); off += len(DATA[d]['y_train'])
    silo = np.array(silo); S = len(doms)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    model = build_bn_model(X.shape[1]) if bn else fl_xp.build_model(X.shape[1], True)
    tv = model.trainable_variables; allw = model.weights
    if not bn: assert len(allw) == len(tv) == 8
    tv_idx = [next(j for j, w in enumerate(allw) if w is v) for v in tv]
    bn_ids = {id(w) for l in model.layers if isinstance(l, layers.BatchNormalization) for w in l.weights}
    local = {j for j, w in enumerate(allw) if id(w) in bn_ids} if bn_local else set()
    K = len(parts); n_k = np.array([len(p) for p in parts], dtype=np.float64)
    gw = [w.numpy().copy() for w in allw]; init_sig = fl_xp.wsig([gw[j] for j in tv_idx])
    sw = {i: [w.copy() for w in gw] for i in range(S)}
    buf = [tf.Variable(np.zeros_like(v.numpy()), trainable=False) for v in tv]
    lossf = tf.keras.losses.SparseCategoricalCrossentropy()
    lr_t, mom_t, lam_t = tf.constant(lr, tf.float32), tf.constant(momentum, tf.float32), tf.constant(lam, tf.float32)
    feat_model = Model(model.input, [hidden_tensor(model, bn), model.output])
    H = 128
    cov_o = tf.Variable(np.zeros((H, H), np.float32), trainable=False); emb_o = tf.Variable(np.zeros(n_rff, np.float32), trainable=False)
    rng = np.random.default_rng(seed * 7919 + 17); sigma = None; Wr = br = None

    @tf.function(reduce_retracing=True)
    def local_epoch(Xc, yc, perm):                                 # identical to the cross-silo grid step
        nb = tf.shape(perm)[0] // batch
        for i in tf.range(nb):
            bi = perm[i * batch:(i + 1) * batch]; xb, yb = tf.gather(Xc, bi), tf.gather(yc, bi)
            with tf.GradientTape() as tape:
                loss = lossf(yb, model(xb, training=True))
            grads = tape.gradient(loss, tv)
            if clip_norm is not None: grads, _ = tf.clip_by_global_norm(grads, clip_norm)
            for w, b, g in zip(tv, buf, grads): b.assign(mom_t * b + g); w.assign_sub(lr_t * b)
        return nb

    def coral_pen(h):
        n = tf.cast(tf.shape(h)[0], tf.float32); hc = h - tf.reduce_mean(h, 0, keepdims=True)
        c = tf.matmul(hc, hc, transpose_a=True) / (n - 1.0)
        return tf.reduce_sum(tf.square(c - cov_o)) / (4.0 * H * H)
    def mmd_pen(h, Wt, bt):
        phi = tf.sqrt(2.0 / n_rff) * tf.cos(tf.matmul(h, Wt) + bt)
        return tf.reduce_sum(tf.square(tf.reduce_mean(phi, 0) - emb_o))

    def make_pen_epoch(Wt, bt):
        @tf.function(reduce_retracing=True)
        def pen_epoch(Xc, yc, perm):
            nb = tf.shape(perm)[0] // batch; tot = tf.constant(0.0)
            for i in tf.range(nb):
                bi = perm[i * batch:(i + 1) * batch]; xb, yb = tf.gather(Xc, bi), tf.gather(yc, bi)
                with tf.GradientTape() as tape:
                    h, p = feat_model(xb, training=True)
                    pen = coral_pen(h) if method == 'coral' else mmd_pen(h, Wt, bt)
                    loss = lossf(yb, p) + lam_t * pen
                grads = tape.gradient(loss, tv)
                if clip_norm is not None: grads, _ = tf.clip_by_global_norm(grads, clip_norm)
                for w, b, g in zip(tv, buf, grads): b.assign(mom_t * b + g); w.assign_sub(lr_t * b)
                tot += pen
            return tot / tf.cast(tf.maximum(nb, 1), tf.float32)
        return pen_epoch

    def silo_stats(i):                                             # aggregate statistics of silo i under silo i's model
        for v, w in zip(allw, sw[i]): v.assign(w)
        h = feat_model.predict(DATA[doms[i]]['X_train'], batch_size=8192, verbose=0)[0].astype(np.float64)
        out = {'cov': np.cov(h, rowvar=False).astype(np.float32)}
        if method == 'mmd': out['emb'] = (np.sqrt(2.0 / n_rff) * np.cos(h @ Wr + br)).mean(0).astype(np.float32)
        return out, h

    XV = np.concatenate([DATA[d]['X_val'] for d in doms]); XT = np.concatenate([DATA[d]['X_test'] for d in doms])
    CV = np.cumsum([0] + [len(DATA[d]['y_val']) for d in doms]); CT = np.cumsum([0] + [len(DATA[d]['y_test']) for d in doms])
    gpu = {k: (tf.constant(X[p]), tf.constant(y[p].astype(np.int32))) for k, p in enumerate(parts)}
    hist, pens = [], []; t0 = time.time(); pen_epoch = None
    if method == 'mmd':                                            # bandwidth: mean of each silo's own median pairwise distance (initial model)
        meds = []
        for i in range(S):
            for v, w in zip(allw, sw[i]): v.assign(w)
            h = feat_model.predict(DATA[doms[i]]['X_train'][rng.choice(len(DATA[doms[i]]['X_train']), 1000, replace=False)], verbose=0)[0]
            dd = np.sqrt(((h[:, None, :] - h[None, :, :]) ** 2).sum(-1)); meds.append(float(np.median(dd[np.triu_indices(1000, 1)])))
        sigma = max(float(np.mean(meds)), 1e-3)
        Wr = (rng.standard_normal((H, n_rff)) / sigma).astype(np.float32); br = rng.uniform(0, 2 * np.pi, n_rff).astype(np.float32)
    if method != 'fedavg': pen_epoch = make_pen_epoch(tf.constant(Wr if Wr is not None else np.zeros((H, n_rff), np.float32)),
                                                       tf.constant(br if br is not None else np.zeros(n_rff, np.float32)))
    for r in range(1, rounds + 1):
        st = [silo_stats(i)[0] for i in range(S)] if method != 'fedavg' else None
        local_ws, rp = [], []
        for k in range(K):
            for v, w in zip(allw, sw[silo[k]]): v.assign(w)
            for b in buf: b.assign(tf.zeros_like(b))
            Xc, yc = gpu[k]
            perm = tf.constant(np.random.default_rng(seed * 100003 + r * 101 + k * 7 + 0).permutation(int(n_k[k])), tf.int32)
            if method == 'fedavg': local_epoch(Xc, yc, perm)
            else:
                o = 1 - silo[k]; cov_o.assign(st[o]['cov'])
                if method == 'mmd': emb_o.assign(st[o]['emb'])
                rp.append(float(pen_epoch(Xc, yc, perm)))
            local_ws.append([v.numpy().copy() for v in allw])
        a = n_k / n_k.sum()
        shared = [sum(a[k] * local_ws[k][L] for k in range(K)) for L in range(len(allw))]
        for i in range(S):
            if local:
                ai = np.where(silo == i, n_k, 0.0); ai = ai / ai.sum()
                loc = [sum(ai[k] * local_ws[k][L] for k in range(K)) for L in range(len(allw))]
                sw[i] = [loc[L] if L in local else shared[L] for L in range(len(allw))]
            else: sw[i] = [w for w in shared]
        if rp: pens.append(float(np.mean(rp)))
        if r == 1 or r % eval_every == 0 or r == rounds:
            rec = {'round': r}
            if rp: rec['penalty'] = pens[-1]
            for i, d in enumerate(doms):
                for v, w in zip(allw, sw[i]): v.assign(w)
                pv = model.predict(XV, batch_size=8192, verbose=0).argmax(1)[CV[i]:CV[i + 1]]
                pt = model.predict(XT, batch_size=8192, verbose=0).argmax(1)[CT[i]:CT[i + 1]]
                rec[f'val_{d}'] = mf1(DATA[d]['y_val'], pv); rec[f'test_{d}'] = mf1(DATA[d]['y_test'], pt)
                rec[f'test_pc_{d}'] = pcf(DATA[d]['y_test'], pt); rec[f'npred_{d}'] = int(len(np.unique(pt)))
                if r == rounds: rec[f'cm_{d}'] = confusion_matrix(DATA[d]['y_test'], pt, labels=LAB).tolist()
            hist.append(rec)
            if log: log(r, rec)
            if not all(np.isfinite(w).all() for i in range(S) for w in sw[i]): return {'diverged': True, 'history': hist}
    del gpu, model, feat_model; tf.keras.backend.clear_session(); gc.collect()
    return {'diverged': False, 'history': hist, 'minutes': (time.time() - t0) / 60, 'init_sig': init_sig,
            'partition_sig': fl_xp.partition_sig(parts), 'client_sizes': n_k.astype(int).tolist(), 'setting': setting,
            'method': method, 'lam': lam, 'bn': bn, 'bn_local': bn_local, 'lr': lr, 'seed': seed, 'sigma': sigma,
            'penalty_per_round': pens, 'n_local_tensors': len(local)}
'''
open(f'{AL}/code/align_core.py', 'w').write(ALIGN_CORE); open('/content/align_core.py', 'w').write(ALIGN_CORE)
sys.path.insert(0, '/content')
import fl_xp, align_core; importlib.reload(fl_xp); importlib.reload(align_core)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xp.N_CLASSES)
NC = 5
def load(dom, sp):                     # loader of the cross-silo notebook, unchanged
    Z = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = Z[f'y_{k}'] < NC
        out[f'X_{k}'] = Z[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = Z[f'y_{k}'][m].astype(np.int32)
    return out
DATA = {d: load(d, 'random') for d in ('iot', 'ent')}
LRX = LR['xsilo']['fedavg']
L3 = lambda o, k: float(np.mean([h[k] for h in o['history'] if h['round'] in (80, 90, 100)]))

# ---- Step 0: plain FedAvg through the new harness must reproduce the cross-silo grid run (seed 42) exactly ----
fp0 = f'{AL}/check/fedavg_xsilo_s42.json'
if not os.path.exists(fp0): json.dump(align_core.run_align(DATA, fl_xp, 'xsilo', 'fedavg', 42, LRX), open(fp0, 'w'))
o = json.load(open(fp0)); ref = json.load(open(f'{XS}/grid/random_dirichlet0.5_xsilo_fedavg_lr{LRX}_s42.json'))
d0 = max(abs(a[f'{k}_{d}'] - b[f'{k}_{d}']) for a, b in zip(o['history'], ref['history']) for d in ('iot', 'ent') for k in ('val', 'test'))
same = (len(o['history']) == len(ref['history']), o['client_sizes'] == ref['client_sizes'], o['partition_sig'] == ref['partition_sig'], o['init_sig'] == ref['init_sig'])
print(f'Step 0 reproduction (new harness vs cross-silo grid, seed 42, all evaluations): max |diff| = {d0:.6f} | rounds/clients/partition/init identical: {same}')
assert d0 < 1e-3 and all(same), 'harness does not reproduce the grid - STOP and paste this output to Claude'

# ---- Step 1: smoke test of every variant (3 rounds, seed 42; not used for any result) ----
for tag, kw in (('coral_lam1', dict(setting='xsilo', method='coral', lam=1.0)), ('mmd_lam1', dict(setting='xsilo', method='mmd', lam=1.0)),
                ('bn_shared', dict(setting='xsilo', method='fedavg', bn=True)), ('fedbn_local', dict(setting='xsilo', method='fedavg', bn=True, bn_local=True)),
                ('bn_within_iot', dict(setting='iot', method='fedavg', bn=True))):
    t0 = time.time(); r = align_core.run_align(DATA, fl_xp, seed=42, lr=LRX, rounds=3, eval_every=1, **kw)
    h = r['history'][-1]; doms = ('iot', 'ent') if kw['setting'] == 'xsilo' else (kw['setting'],)
    print(f'  {tag:<14} finite={not r["diverged"]} | ' + ' | '.join(f'{d} test {h["test_" + d]:.4f}' for d in doms)
          + (f' | penalty r1..r3 {[round(p, 5) for p in r["penalty_per_round"]]}' if r['penalty_per_round'] else '')
          + (f' | sigma {r["sigma"]:.3f}' if r['sigma'] else '') + f' | silo-local tensors {r["n_local_tensors"]} | {time.time() - t0:.0f} s', flush=True)
    json.dump(r, open(f'{AL}/smoke/{tag}.json', 'w'))
print('harness saved to results/align/code/align_core.py | CELL 1 DONE')

In [ ]:
# CROSSSILO_ALIGN — CELL 2: pilot on VALIDATION only (seed 42, random split, Dirichlet 0.5). GPU T4, ~1.5 h, resumable.
#   Chooses (a) the penalty weight lam for CORAL and MMD (lr fixed at the cross-silo FedAvg lr) and
#   (b) the learning rate for the batch-norm models (BN shared, FedBN silo-local, BN within each silo).
#   Rule (same as the cross-silo lr pilot): mean validation macro-F1 over rounds 80/90/100; cross-silo = mean of the two
#   silos; differences < 0.0005 go to the smaller value. Test scores are computed by the harness but DISCARDED unseen.
#   Output: results/align/pilot/*.json and results/align/align_chosen.json. Prints relative paths only.
import os, glob, json, time, sys, importlib
import numpy as np, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; AL = f'{DRIVE_PATH}/results/align'; PL = f'{AL}/pilot'; os.makedirs(PL, exist_ok=True)
assert os.path.exists(f'{AL}/check/fedavg_xsilo_s42.json'), 'run Cell 1 first'
src = open(f'{V2D}/fl_core.py').read(); open('/content/fl_xp.py', 'w').write(src.replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/align_core.py', 'w').write(open(f'{AL}/code/align_core.py').read()); sys.path.insert(0, '/content')
import fl_xp, align_core; importlib.reload(fl_xp); importlib.reload(align_core)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'))
NC = 5
def load(dom, sp):
    Z = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = Z[f'y_{k}'] < NC
        out[f'X_{k}'] = Z[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = Z[f'y_{k}'][m].astype(np.int32)
    return out
DATA = {d: load(d, 'random') for d in ('iot', 'ent')}
LRX = json.load(open(f'{XS}/lr_chosen.json'))['xsilo']['fedavg']
L3 = lambda o, k: float(np.mean([h[k] for h in o['history'] if h['round'] in (80, 90, 100)]))

JOBS = ([('coral', 'xsilo', dict(method='coral', lam=l, lr=LRX)) for l in (0.01, 0.1, 1.0, 10.0)] +
        [('mmd', 'xsilo', dict(method='mmd', lam=l, lr=LRX)) for l in (0.1, 1.0, 10.0, 100.0)] +
        [(name, st, dict(method='fedavg', bn=True, bn_local=loc, lr=lr))
         for name, st, loc in (('bn_shared', 'xsilo', False), ('fedbn', 'xsilo', True), ('bn_iot', 'iot', False), ('bn_ent', 'ent', False))
         for lr in (0.1, 0.3)])
tag = lambda n, st, kw: f'{n}_{st}_' + (f'lam{kw["lam"]}' if kw['method'] != 'fedavg' else f'lr{kw["lr"]}') + '_s42'
todo = [j for j in JOBS if not os.path.exists(f'{PL}/{tag(*j)}.json')]
print(f'{len(JOBS)} pilot runs | {len(JOBS) - len(todo)} done | {len(todo)} to run')
t0 = time.time()
for i, (n, st, kw) in enumerate(todo):
    r = align_core.run_align(DATA, fl_xp, st, seed=42, **kw)
    for h in r['history']:                                         # discard test scores unseen
        for k in [k for k in h if k.startswith(('test', 'cm_', 'npred'))]: del h[k]
    r['pilot'] = n; json.dump(r, open(f'{PL}/{tag(n, st, kw)}.json', 'w'))
    doms = ('iot', 'ent') if st == 'xsilo' else (st,)
    print(f'  [{i + 1}/{len(todo)}] {tag(n, st, kw):<28} ' + ' | '.join(f'val {d} {L3(r, "val_" + d):.4f}' for d in doms)
          + (' DIVERGED' if r['diverged'] else '') + f' | {r.get("minutes", 0):.1f} min', flush=True)

# ---- selection ----
rows = []
for f in sorted(glob.glob(f'{PL}/*.json')):
    r = json.load(open(f)); doms = ('iot', 'ent') if r['setting'] == 'xsilo' else (r['setting'],)
    val = np.nan if r['diverged'] else float(np.mean([L3(r, f'val_{d}') for d in doms]))
    rows.append((r['pilot'], r['lam'] if r['method'] != 'fedavg' else r['lr'], val, r.get('minutes') or np.nan))
chosen = {}
print('\n===== pilot: validation macro-F1, mean of rounds 80/90/100 (cross-silo = mean of both silos) =====')
for name in ('coral', 'mmd', 'bn_shared', 'fedbn', 'bn_iot', 'bn_ent'):
    g = sorted([x for x in rows if x[0] == name], key=lambda x: x[1])
    for x in g: print(f'  {name:<10} {"lam" if name in ("coral", "mmd") else "lr":<3} = {x[1]:<6} val {x[2]:.4f} | {x[3]:.1f} min')
    ok = [x for x in g if np.isfinite(x[2])]; best = max(x[2] for x in ok)
    chosen[name] = float(next(x[1] for x in ok if x[2] >= best - 0.0005))       # near-ties go to the smaller value
json.dump({'rule': 'seed 42, random split, Dirichlet 0.5, validation only; mean val macro-F1 over rounds 80/90/100; '
                   'cross-silo = mean of the two silos; differences < 0.0005 go to the smaller value',
           'lr_alignment': LRX, 'chosen': chosen}, open(f'{AL}/align_chosen.json', 'w'), indent=1)
print('\nchosen:', chosen)
mins = {n: np.nanmean([x[3] for x in rows if x[0] == n]) for n in chosen}
print('mean minutes per 100-round run:', {k: round(v, 1) for k, v in mins.items()})
print(f'saved results/align/align_chosen.json | CELL 2 DONE in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# CROSSSILO_ALIGN — CELL 3: MAIN GRID (random split, Dirichlet 0.5, 10 seeds). GPU T4, ~7.2 h, fully resumable.
#   Settings chosen in Cell 2 on validation data (results/align/align_chosen.json). Order = priority, so a partial grid
#   is still usable: FedBN, CORAL, BN within each silo (baselines for the BN models), BN shared, MMD (slowest) last.
#   Paired comparisons in the summary (same clients, partitions and seeds; test macro-F1, mean of rounds 80/90/100):
#     CORAL, MMD           vs within-silo FedAvg (FA-MLP grid) and vs cross-silo FedAvg (FA-MLP grid)
#     FedBN, BN shared     vs BN within each silo (new runs) ; FedBN also vs BN shared
#   Output: results/align/grid/*.json. Prints relative paths only.
import os, glob, json, time, sys, importlib
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
from scipy import stats
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; AL = f'{DRIVE_PATH}/results/align'; GR = f'{AL}/grid'; os.makedirs(GR, exist_ok=True)
CH = json.load(open(f'{AL}/align_chosen.json')); C = CH['chosen']; LRX = CH['lr_alignment']
src = open(f'{V2D}/fl_core.py').read(); open('/content/fl_xp.py', 'w').write(src.replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/align_core.py', 'w').write(open(f'{AL}/code/align_core.py').read()); sys.path.insert(0, '/content')
import fl_xp, align_core; importlib.reload(fl_xp); importlib.reload(align_core)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| chosen:', C)
NC = 5
def load(dom, sp):
    Z = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = Z[f'y_{k}'] < NC
        out[f'X_{k}'] = Z[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = Z[f'y_{k}'][m].astype(np.int32)
    return out
DATA = {d: load(d, 'random') for d in ('iot', 'ent')}
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
CFG = {'fedbn':     ('xsilo', dict(method='fedavg', bn=True, bn_local=True, lr=C['fedbn'])),
       'coral':     ('xsilo', dict(method='coral', lam=C['coral'], lr=LRX)),
       'bn_iot':    ('iot',   dict(method='fedavg', bn=True, lr=C['bn_iot'])),
       'bn_ent':    ('ent',   dict(method='fedavg', bn=True, lr=C['bn_ent'])),
       'bn_shared': ('xsilo', dict(method='fedavg', bn=True, lr=C['bn_shared'])),
       'mmd':       ('xsilo', dict(method='mmd', lam=C['mmd'], lr=LRX))}
EST = {'fedbn': 6.9, 'coral': 7.5, 'bn_iot': 3.5, 'bn_ent': 3.5, 'bn_shared': 7.0, 'mmd': 14.7}
JOBS = [(n, s) for n in CFG for s in SEEDS]
fp = lambda n, s: f'{GR}/{n}_s{s}.json'
todo = [j for j in JOBS if not os.path.exists(fp(*j))]
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{sum(EST[n] for n, _ in todo) / 60:.1f} h left')
L3 = lambda o, k: float(np.mean([h[k] for h in o['history'] if h['round'] in (80, 90, 100)]))
t0 = time.time()
for i, (n, s) in enumerate(todo):
    st, kw = CFG[n]; r = align_core.run_align(DATA, fl_xp, st, seed=s, **kw); r['config'] = n
    json.dump(r, open(fp(n, s), 'w')); doms = ('iot', 'ent') if st == 'xsilo' else (st,)
    print(f'  [{i + 1}/{len(todo)}] {n:<10} s{s:<5} ' + ' | '.join(f'test {d} {L3(r, "test_" + d):.4f}' for d in doms)
          + (' DIVERGED' if r['diverged'] else '') + f' | {r.get("minutes", 0):.1f} min | ~{sum(EST[x] for x, _ in todo[i + 1:]) / 60:.1f} h left', flush=True)

# ---- summary (works on a partial grid) ----
def vec(kind, n, d):
    out = []
    for s in SEEDS:
        if kind == 'new': f = fp(n, s)
        elif kind == 'within': f = f'{XS}/grid/random_dirichlet0.5_{d}_fedavg_lr{json.load(open(f"{XS}/lr_chosen.json"))[d]["fedavg"]}_s{s}.json'
        else: f = f'{XS}/grid/random_dirichlet0.5_xsilo_fedavg_lr{LRX}_s{s}.json'
        out.append(L3(json.load(open(f)), f'test_{d}') if os.path.exists(f) else np.nan)
    return np.array(out)
def pr(a, b, lab):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 2: return f'{lab}: n<2'
    d = 100 * (b[m] - a[m]); se = d.std(ddof=1) / np.sqrt(len(d)); t = stats.t.ppf(.975, len(d) - 1)
    return (f'{lab}: {a[m].mean():.4f} -> {b[m].mean():.4f} = {d.mean():+.2f} pp [{d.mean() - t * se:+.2f}, {d.mean() + t * se:+.2f}] '
            f'(n={len(d)}, lower in {int((d < 0).sum())}, p {stats.ttest_rel(b[m], a[m]).pvalue:.4f})')
print('\n===== test macro-F1, random split, Dirichlet 0.5, paired over seeds (pp) =====')
for d in ('iot', 'ent'):
    print(f'--- {d} silo')
    for n in ('coral', 'mmd'):
        v = vec('new', n, d)
        print(f'  {n:<10} ' + pr(vec('within', None, d), v, 'vs within FedAvg') + '\n' + ' ' * 13 + pr(vec('xsilo', None, d), v, 'vs cross-silo FedAvg'))
    bw = vec('new', f'bn_{d}', d)
    for n in ('fedbn', 'bn_shared'):
        print(f'  {n:<10} ' + pr(bw, vec('new', n, d), 'vs BN within'))
    print(f'  {"fedbn":<10} ' + pr(vec('new', 'bn_shared', d), vec('new', 'fedbn', d), 'vs BN shared'))
done = sum(os.path.exists(fp(*j)) for j in JOBS)
print(f'\nruns finished {done}/{len(JOBS)} | session {(time.time() - t0) / 60:.1f} min' + (' | CELL 3 DONE' if done == len(JOBS) else ' | grid not complete - rerun this cell'))

In [ ]:
# CROSSSILO_ALIGN — CELL 4: EXPORT (any runtime, CPU is enough; ~1 min). Trains nothing.
#   Zips everything under results/align (harness code, checks, smoke test, pilot, chosen settings, 60 grid runs) -> download.
#   Prints relative paths only.
import os, glob, zipfile, json
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; AL = f'{cands[0]}/results/align'
n_grid = len(glob.glob(f'{AL}/grid/*.json')); assert n_grid == 60, f'grid has {n_grid}/60 runs - rerun Cell 3 first'
zp = f'{AL}/export_align.zip'; n = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{AL}/**/*', recursive=True)):
        if os.path.isfile(f) and not f.endswith('.zip'): z.write(f, os.path.relpath(f, AL)); n += 1
subdirs = sorted({os.path.relpath(f, AL).split('/')[0] for f in glob.glob(f'{AL}/**/*', recursive=True) if os.path.isfile(f) and not f.endswith('.zip')})
print(f'exported {n} files ({os.path.getsize(zp) / 1e6:.1f} MB) from {subdirs} -> results/align/export_align.zip | CELL 4 DONE'); files.download(zp)